# FusionFlow: combining the streams

For every day, a velocity anomaly and a temperature anomaly summarise the forecasts of the preceding 30 days, the same window that defines a GLOF label:
- **velocity:** the 90th percentile over all pixel-days of ln((v + 10) / (v_typ + 10)), where v is the TerraFlow forecast and v_typ the pixel's median speed in the same calendar month over its training years (to 2020), both in m/yr;
- **temperature:** the median over all pixel-days of the TempFlow forecast minus the pixel's seasonal cycle before 7 June 2018, used up to the end of 2022 because Terra's overpass drifts earlier afterwards.

A logistic regression of the two anomalies (L2 penalty with C = 1, balanced class weights, standardised inputs, missing values set to the training mean) is trained on every day of the training years, each day weighted 1/31 because neighbouring days share most of their window. FusionFlow adds the log-odds of RiskFlow and of this tabular stream, with both probabilities kept within 0.01 and 0.99, and alerts at 0.5. It is compared with RiskFlow alone, the tabular stream alone, the agreement rule (both at least 0.5) and the either-stream rule (either at least 0.5), with each year held out in turn and in the main split. All settings were fixed before the first run.

**Inputs:** `RiskFlow_Shisper/riskflow_predictions.csv` (notebook 10; the multi-site model of notebook 12 is used instead only if it met its criteria), `TerraFlow_Shisper/terraflow_forecasts_all.npz` (notebook 20) and `TempFlow_Shisper/tempflow_forecasts_all.npz` (notebook 30). **Output:** `FusionFlow_Shisper`. CPU runtime, a few minutes.

## Setup

In [ ]:
import os, json
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from matplotlib import dates as mdates

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
RF_SINGLE = f'{DRIVE}/RiskFlow_Shisper/riskflow_predictions.csv'
RF_MULTI = f'{DRIVE}/RiskFlow_multisite_results'
TF_NPZ = f'{DRIVE}/TerraFlow_Shisper/terraflow_forecasts_all.npz'
TP_NPZ = f'{DRIVE}/TempFlow_Shisper/tempflow_forecasts_all.npz'
OUT = f'{DRIVE}/FusionFlow_Shisper'
os.makedirs(OUT, exist_ok=True)

OUTBURSTS = ['2019-06-22', '2020-05-29', '2021-05-15', '2022-05-07']   # Lovell and Muhammad (2025)
WINDOW = 30                     # days: label window and anomaly window
THRESH = 0.5
CLIP = 0.01                     # probabilities kept within [0.01, 0.99] before pooling
C_REG = 1.0
DAY_W = 1 / (WINDOW + 1)        # neighbouring days share most of their window
VEL_TRAIN_LAST = '2020-12-31'   # TerraFlow's training years
VEL_OFFSET, VEL_Q, VEL_MIN_N = 10.0, 0.9, 50
TEMP_CUT, TEMP_LAST, TEMP_MIN_N = '2018-06-07', '2022-12-31', 20
MAIN_TRAIN_LAST_YEAR = 2021
EPOCH = pd.Timestamp('1970-01-01')
def dnum(s):
    return int((pd.Timestamp(s) - EPOCH).days)
SETTINGS = dict(window_days=WINDOW, threshold=THRESH, clip=CLIP, C=C_REG, day_weight=round(DAY_W, 4), velocity_quantile=VEL_Q,
                velocity_offset_m_per_yr=VEL_OFFSET, velocity_typical_years=f'to {VEL_TRAIN_LAST}',
                temperature_climatology=f'before {TEMP_CUT}', temperature_used_until=TEMP_LAST)

## RiskFlow probabilities

In [ ]:
mj, mc = f'{RF_MULTI}/multisite_results.json', f'{RF_MULTI}/shisper_predictions_multisite.csv'
if os.path.exists(mj) and os.path.exists(mc) and json.load(open(mj)).get('criteria_met', False):
    R = pd.read_csv(mc).rename(columns={'p_loyo': 'p_rf_loyo', 'p_main': 'p_rf_main'})
    RF_VERSION = 'RiskFlow, multi-site'
else:
    R = pd.read_csv(RF_SINGLE).rename(columns={'p_loyo': 'p_rf_loyo', 'p_main': 'p_rf_main'})
    RF_VERSION = 'RiskFlow, Shisper only'
R = R[['date', 'label', 'p_rf_loyo', 'p_rf_main']].copy()
R['date'] = pd.to_datetime(R['date'])
R['day'] = (R.date - EPOCH).dt.days
R['year'] = R.date.dt.year
R['fold'] = R.year.astype(str).where(R.year > 2019, '2018-19')
OB = np.array([dnum(o) for o in OUTBURSTS])
def label_of(days):
    d = np.asarray(days)[:, None] - OB[None, :]
    return ((d >= 0) & (d <= WINDOW)).any(1).astype(int)
assert (label_of(R.day) == R.label.values).all(), 'RiskFlow labels differ from the outburst dates'
print(RF_VERSION, f'| {len(R)} scenes, {int(R.label.sum())} GLOF')

## Velocity anomaly from TerraFlow

In [ ]:
DAYS = np.arange(R.day.min(), R.day.max() + 1)                     # every day of the scene archive
def window_stat(day_sorted, val_sorted, fn, min_n):
    lo = np.searchsorted(day_sorted, DAYS - WINDOW, 'left'); hi = np.searchsorted(day_sorted, DAYS, 'right')
    out = np.full(len(DAYS), np.nan)
    for k in np.nonzero(hi - lo >= min_n)[0]:
        out[k] = fn(val_sorted[lo[k]:hi[k]])
    return out, hi - lo

if os.path.exists(TF_NPZ):
    V = np.load(TF_NPZ)
    vpix, vday = V['pix'].astype('int64'), V['day'].astype('int64')
    vobs, vfc = V['observed'].astype('float64'), np.maximum(V['forecast'].astype('float64'), 0.0)
    vmon = vday.astype('datetime64[D]').astype('datetime64[M]').astype('int64') % 12
    tr = vday <= dnum(VEL_TRAIN_LAST)
    key = vpix * 12 + vmon
    gm = pd.Series(vobs[tr]).groupby(key[tr]).agg(['median', 'size'])
    typ = np.array(pd.Series(key).map(gm['median'][gm['size'] >= 10]), dtype='float64')   # pixel and calendar month
    miss = np.isnan(typ)
    typ[miss] = np.array(pd.Series(vpix[miss]).map(pd.Series(vobs[tr]).groupby(vpix[tr]).median()), dtype='float64')
    typ[np.isnan(typ)] = np.median(vobs[tr])
    va = np.log((vfc + VEL_OFFSET) / (typ + VEL_OFFSET))
    o = np.argsort(vday, kind='stable')
    VEL, VEL_N = window_stat(vday[o], va[o], lambda a: np.quantile(a, VEL_Q), VEL_MIN_N)
    print(f'TerraFlow: {len(vday):,} pixel-days | velocity anomaly on {np.isfinite(VEL).mean():.0%} of days')
    del V, vpix, vobs, vfc, vmon, key, typ, va
else:
    VEL, VEL_N = np.full(len(DAYS), np.nan), np.zeros(len(DAYS), int)
    print('TerraFlow forecasts not found: velocity left out')

## Temperature anomaly from TempFlow

In [ ]:
if os.path.exists(TP_NPZ):
    T = np.load(TP_NPZ)
    fpix, fday, fc = T['fc_pix'].astype(int), T['fc_day'].astype('int64'), T['forecast'].astype('float64')
    opix, oday, ob = T['obs_pix'].astype(int), T['obs_day'].astype('int64'), T['observed'].astype('float64')
    npix = len(T['pix_lat'])
    def harm(days):
        w = 2 * np.pi * pd.DatetimeIndex(days.astype('datetime64[D]')).dayofyear.values / 365.25
        return np.column_stack([np.ones(len(w)), np.sin(w), np.cos(w), np.sin(2 * w), np.cos(2 * w)])
    trn = oday < dnum(TEMP_CUT)
    pooled = np.linalg.lstsq(harm(oday[trn]), ob[trn], rcond=None)[0]
    COEF = np.vstack([np.linalg.lstsq(harm(oday[trn & (opix == k)]), ob[trn & (opix == k)], rcond=None)[0]
                      if (trn & (opix == k)).sum() >= 50 else pooled for k in range(npix)])
    ta = fc - (harm(fday) * COEF[fpix]).sum(1)
    keep = fday <= dnum(TEMP_LAST)
    o = np.argsort(fday[keep], kind='stable')
    TEMP, TEMP_N = window_stat(fday[keep][o], ta[keep][o], np.median, TEMP_MIN_N)
    TEMP[DAYS > dnum(TEMP_LAST)] = np.nan
    print(f'TempFlow: {len(fc):,} pixel-day forecasts on {npix} pixels | temperature anomaly on '
          f'{np.isfinite(TEMP[DAYS <= dnum(TEMP_LAST)]).mean():.0%} of days up to {TEMP_LAST}')
else:
    TEMP, TEMP_N = np.full(len(DAYS), np.nan), np.zeros(len(DAYS), int)
    print('TempFlow forecasts not found: temperature left out')

## Tabular stream: daily logistic regression

In [ ]:
D = pd.DataFrame({'day': DAYS, 'vel_anom': VEL, 'vel_n': VEL_N, 'temp_anom': TEMP, 'temp_n': TEMP_N})
D['date'] = EPOCH + pd.to_timedelta(D.day, unit='D')
D['year'] = D.date.dt.year
D['fold'] = D.year.astype(str).where(D.year > 2019, '2018-19')
D['label'] = label_of(D.day)
FEAT = [c for c in ['vel_anom', 'temp_anom'] if D[c].notna().any()]
assert FEAT, 'neither TerraFlow nor TempFlow forecasts were found'

def tab_model():
    return make_pipeline(SimpleImputer(strategy='mean', keep_empty_features=True), StandardScaler(),
                         LogisticRegression(C=C_REG, class_weight='balanced', max_iter=1000))

D['p_tab_loyo'] = np.nan
COEFS = []
for f in sorted(D.fold.unique()):
    te = (D.fold == f).values
    m = tab_model().fit(D.loc[~te, FEAT], D.label[~te], logisticregression__sample_weight=np.full((~te).sum(), DAY_W))
    D.loc[te, 'p_tab_loyo'] = m.predict_proba(D.loc[te, FEAT])[:, 1]
    COEFS.append(dict(model=f'year {f} held out', **{c: round(float(b), 3) for c, b in zip(FEAT, m[-1].coef_[0])},
                      intercept=round(float(m[-1].intercept_[0]), 3)))
trm, tem = (D.year <= MAIN_TRAIN_LAST_YEAR).values, (D.year > MAIN_TRAIN_LAST_YEAR).values
m = tab_model().fit(D.loc[trm, FEAT], D.label[trm], logisticregression__sample_weight=np.full(trm.sum(), DAY_W))
D['p_tab_main'] = np.nan
D.loc[tem, 'p_tab_main'] = m.predict_proba(D.loc[tem, FEAT])[:, 1]
COEFS.append(dict(model='main split (train to 2021)', **{c: round(float(b), 3) for c, b in zip(FEAT, m[-1].coef_[0])},
                  intercept=round(float(m[-1].intercept_[0]), 3)))
COEFS = pd.DataFrame(COEFS)

# every day, not only scene days (years held out)
DAILY_AUC = round(float(roc_auc_score(D.label, D.p_tab_loyo)), 3)
UNI_AUC = {c: round(float(roc_auc_score(D.label[D[c].notna()], D[c][D[c].notna()])), 3) for c in FEAT}
before = np.zeros(len(D), bool)
EV = []
for o, name in zip(OB, OUTBURSTS):
    a = (D.day >= o) & (D.day <= o + WINDOW); b = (D.day >= o - WINDOW) & (D.day < o)
    before |= b.values
    hit = D[a & (D.p_tab_loyo >= THRESH)]
    EV.append(dict(outburst=name, days_flagged_in_30_after=f'{len(hit)} of {int(a.sum())}',
                   days_flagged_in_30_before=f'{int((b & (D.p_tab_loyo >= THRESH)).sum())} of {int(b.sum())}',
                   mean_p_after=round(float(D.p_tab_loyo[a].mean()), 3), mean_p_before=round(float(D.p_tab_loyo[b].mean()), 3)))
EV = pd.DataFrame(EV)
quiet = (D.label == 0).values & ~before
QUIET_FA = round(float((D.p_tab_loyo[quiet] >= THRESH).mean()), 3)
print('daily AUC, years held out:', DAILY_AUC, '| single anomaly:', UNI_AUC)
print(f'days flagged away from any outburst (not within 30 days before or after): {QUIET_FA:.1%}')
print(COEFS.to_string(index=False))
print(EV.to_string(index=False))

## Scene-level comparison

In [ ]:
def pool(a, b):
    a = np.clip(np.asarray(a, float), CLIP, 1 - CLIP); b = np.clip(np.asarray(b, float), CLIP, 1 - CLIP)
    return 1 / (1 + np.exp(-(np.log(a / (1 - a)) + np.log(b / (1 - b)))))

S = R.merge(D[['day', 'vel_anom', 'vel_n', 'temp_anom', 'temp_n', 'p_tab_loyo', 'p_tab_main']], on='day', how='left')
S['days_since_outburst'] = [min([d - o for o in OB if d >= o], default=np.nan) for d in S.day]
for s in ['loyo', 'main']:
    rf, tab, has = S[f'p_rf_{s}'], S[f'p_tab_{s}'], S[f'p_rf_{s}'].notna()
    S[f'p_fusion_{s}'] = pool(rf, tab)
    S[f'agree_{s}'] = ((rf >= THRESH) & (tab >= THRESH)).astype(float).where(has)
    S[f'either_{s}'] = ((rf >= THRESH) | (tab >= THRESH)).astype(float).where(has)

def wilson(k, n, z=1.96):
    # 95 % Wilson interval for k of n
    if n == 0:
        return ''
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return f'{max(0.0, c - w):.2f} to {min(1.0, c + w):.2f}'

def summ(y, p, binary=False):
    y, p = np.asarray(y, int), np.asarray(p, float); k = ~np.isnan(p); y, p = y[k], p[k]
    pr = p >= THRESH
    TP, FN = int((pr & (y == 1)).sum()), int((~pr & (y == 1)).sum()); FP, TN = int((pr & (y == 0)).sum()), int((~pr & (y == 0)).sum())
    return dict(caught=f'{TP} of {TP + FN}', recall_ci95=wilson(TP, TP + FN), false_alarms=f'{FP} of {FP + TN}',
                far_ci95=wilson(FP, FP + TN), auc=None if binary or len(set(y)) < 2 else round(float(roc_auc_score(y, p)), 3))

METHODS = [('RiskFlow alone', 'p_rf'), ('Tabular stream alone', 'p_tab'), ('Agreement rule', 'agree'),
           ('Either-stream rule', 'either'), ('FusionFlow', 'p_fusion')]
TAB = {s: pd.DataFrame({name: summ(S.label, S[f'{col}_{s}'], binary=col in ('agree', 'either')) for name, col in METHODS}).T
       for s in ['loyo', 'main']}
DISAGREE = {}
for s, title in [('loyo', 'Years held out in turn (all scenes)'), ('main', 'Main split (test 2022 onward)')]:
    has = S[f'p_rf_{s}'].notna()
    d = has & ((S[f'p_rf_{s}'] >= THRESH) != (S[f'p_tab_{s}'] >= THRESH))
    DISAGREE[s] = dict(scenes=int(d.sum()), of_which_glof=int((d & (S.label == 1)).sum()), of_scenes=int(has.sum()))
    print(title); print(TAB[s].to_string())
    print(f"streams disagree on {DISAGREE[s]['scenes']} of {DISAGREE[s]['of_scenes']} scenes "
          f"({DISAGREE[s]['of_which_glof']} of them GLOF scenes)\n")
GL = S[S.label == 1][['date', 'days_since_outburst', 'p_rf_loyo', 'p_tab_loyo', 'p_fusion_loyo', 'vel_anom', 'temp_anom']]
print('GLOF scenes (years held out):')
print(GL.assign(date=GL.date.dt.strftime('%Y-%m-%d')).round(3).to_string(index=False))

## Figures

In [ ]:
OBD = [EPOCH + pd.Timedelta(days=int(o)) for o in OB]
def mark(ax, lo=None, hi=None):
    for o in OBD:
        if (lo is None or o >= lo) and (hi is None or o <= hi):
            ax.axvspan(o, o + pd.Timedelta(days=WINDOW), color='#f2c14e', alpha=0.25, lw=0, zorder=0)
            ax.axvline(o, color='0.35', ls='--', lw=1, zorder=1)

lo, hi = pd.Timestamp('2022-03-01'), pd.Timestamp('2022-08-31')
d22, s22 = D[(D.date >= lo) & (D.date <= hi)], S[(S.date >= lo) & (S.date <= hi)]
fig, ax = plt.subplots(3, 1, figsize=(8, 7), sharex=True, gridspec_kw=dict(hspace=0.12))
ax[0].plot(d22.date, d22.vel_anom, c='k', lw=1.2); ax[0].set_ylabel('velocity anomaly\n(log ratio, 90th pct.)')
ax[1].plot(d22.date, d22.temp_anom, c='#2e6b8a', lw=1.2); ax[1].axhline(0, c='0.6', lw=0.8)
ax[1].set_ylabel('temperature\nanomaly (K)')
ax[2].plot(d22.date, d22.p_tab_main, c='#3a6ea5', lw=1.2, label='tabular stream (daily)')
ax[2].plot(s22.date, s22.p_rf_main, '^', c='#b84a2f', ms=7, label='RiskFlow (scene)')
ax[2].plot(s22.date, s22.p_fusion_main, 'o', mfc='none', mec='k', ms=9, mew=1.4, label='FusionFlow (scene)')
ax[2].axhline(THRESH, c='0.4', ls=':', lw=1); ax[2].set_ylim(-0.03, 1.03); ax[2].set_ylabel('probability')
ax[2].legend(fontsize=8, loc='upper right')
for a in ax:
    mark(a, lo, hi)
    for sp in ('top', 'right'):
        a.spines[sp].set_visible(False)
ax[0].set_title('Shisper 2022, main split (models trained to 2021)', fontsize=9.5)
ax[2].xaxis.set_major_locator(mdates.MonthLocator()); ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.savefig(f'{OUT}/fig_fusion_2022.png', dpi=200, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(2, 1, figsize=(12, 5.6), sharex=True, gridspec_kw=dict(hspace=0.1))
ax[0].plot(D.date, D.p_tab_loyo, c='#3a6ea5', lw=0.9); ax[0].axhline(THRESH, c='0.4', ls=':', lw=1)
ax[0].set_ylabel('tabular stream\n(daily)'); ax[0].set_ylim(-0.03, 1.03)
if 'temp_anom' in FEAT:
    ax[0].axvline(pd.Timestamp(TEMP_LAST), c='#2e6b8a', ls=':', lw=1.2)
g = S.label == 1
ax[1].scatter(S.date[~g], S.p_rf_loyo[~g], s=12, c='#9db4cf', label='RiskFlow, other scene')
ax[1].scatter(S.date[~g], S.p_fusion_loyo[~g], s=14, facecolors='none', edgecolors='0.3', lw=0.7, label='FusionFlow, other scene')
ax[1].scatter(S.date[g], S.p_rf_loyo[g], s=50, marker='^', c='#c0392b', label='RiskFlow, GLOF scene')
ax[1].scatter(S.date[g], S.p_fusion_loyo[g], s=80, facecolors='none', edgecolors='k', lw=1.4, label='FusionFlow, GLOF scene')
ax[1].axhline(THRESH, c='0.4', ls=':', lw=1); ax[1].set_ylim(-0.03, 1.03); ax[1].set_ylabel('probability (scene)')
ax[1].legend(fontsize=7.5, ncol=2, loc='upper right')
for a in ax:
    mark(a)
    for sp in ('top', 'right'):
        a.spines[sp].set_visible(False)
ax[0].set_title(f'Years held out in turn, {RF_VERSION}', fontsize=9.5)
plt.savefig(f'{OUT}/fig_fusion_timeline.png', dpi=200, bbox_inches='tight'); plt.show()

## Save

In [ ]:
S.assign(date=S.date.dt.strftime('%Y-%m-%d')).round(4).to_csv(f'{OUT}/fusion_scenes.csv', index=False)
D.assign(date=D.date.dt.strftime('%Y-%m-%d')).round(4).to_csv(f'{OUT}/fusion_daily.csv', index=False)
RES = dict(riskflow=RF_VERSION, streams=FEAT, settings=SETTINGS,
           scene_loyo=TAB['loyo'].to_dict('index'), scene_main=TAB['main'].to_dict('index'), disagreement=DISAGREE,
           daily_auc_loyo=DAILY_AUC, single_anomaly_auc=UNI_AUC, quiet_days_flagged=QUIET_FA,
           events=EV.to_dict('records'), coefficients=COEFS.to_dict('records'),
           availability=dict(velocity=round(float(np.isfinite(D.vel_anom).mean()), 3),
                             temperature_to_2022=round(float(np.isfinite(D.temp_anom[D.day <= dnum(TEMP_LAST)]).mean()), 3)))
json.dump(RES, open(f'{OUT}/fusion_results.json', 'w'), indent=1, default=str)
print('saved in', OUT)